# Word Segmentation with Conditional Random Fields (CRF)
## Ye Kyaw Thu, Language Understanding Lab, Myanmar
## Date: 25 July 2026

CRFSuiteကိုသုံးပြီး Word Segmentation မော်ဒယ်ဆောက်မယ်။ ပြီးတော့ ဆောက်ထားတဲ့ မော်ဒယ်ကို သုံးပြီး မြန်မာစာ စာလုံးတွေကို ဖြတ်ကြည့်ပြီး ရလဒ် ဘယ်လောက် ကောင်းသလဲ ဆိုတာကို လက်တွေ့ လုပ်ကြည့်ကြရအောင်။  

In [1]:
# Confirm which Python interpreter this kernel is using (should be nlp_venv)
import sys
print(sys.executable)

/Users/yadanar/Desktop/AIE-F-B2/nlp_venv/bin/python


## CRFSuite Installation

Link: https://github.com/chokkan/crfsuite   
Library link: https://github.com/chokkan/liblbfgs   
CRFSuite Tutorial link: https://www.chokkan.org/software/crfsuite/tutorial.html

In [2]:
# Check current working directory before creating build folders
%pwd #checking the current working directory in order to ensure that the script is running in the correct environment.

'/Users/yadanar/Desktop/AIE-F-B2/notebooks'

In [3]:
# Create a 'tool' folder to hold cloned build dependencies
!mkdir tool

mkdir: tool: File exists


In [4]:
# Move into the 'tool' folder
%cd tool

/Users/yadanar/Desktop/AIE-F-B2/notebooks/tool


In [5]:
# Confirm the new working directory
%pwd

'/Users/yadanar/Desktop/AIE-F-B2/notebooks/tool'

In [6]:
# Clone the crfsuite source code from GitHub
!git clone https://github.com/chokkan/crfsuite

fatal: destination path 'crfsuite' already exists and is not an empty directory.


In [7]:
# Move into the cloned crfsuite folder
%cd crfsuite/

/Users/yadanar/Desktop/AIE-F-B2/notebooks/tool/crfsuite


In [8]:
# List files to see the crfsuite project layout
!ls

-p              README          config.h.in     depcomp         lib
AUTHORS         aclocal.m4      config.h.in~    doc             libtool
COPYING         autogen.sh      config.log      example         ltmain.sh
ChangeLog       autom4te.cache  config.status   frontend        m4
INSTALL         bench           config.sub      genbinary.sh    missing
Makefile        compile         configure       genbinary.sh.in stamp-h1
Makefile.am     config.guess    configure.in    include         swig
Makefile.in     config.h        crfsuite.sln    install-sh      win32


In [9]:
# Generate ./configure from configure.in (needs autoconf/automake/libtoolize on PATH)
!./autogen.sh

aclocal: warning: autoconf input should be named 'configure.ac', not 'configure.in'
autoheader: warning: autoconf input should be named 'configure.ac', not 'configure.in'
automake: warning: autoconf input should be named 'configure.ac', not 'configure.in'
configure.in:140: warning: 'INCLUDES' is the old name for 'AM_CPPFLAGS' (or '*_CPPFLAGS')
configure.in:140: warning: 'INCLUDES' is the old name for 'AM_CPPFLAGS' (or '*_CPPFLAGS')
automake: warning: autoconf input should be named 'configure.ac', not 'configure.in'
configure.in:140: warning: 'INCLUDES' is the old name for 'AM_CPPFLAGS' (or '*_CPPFLAGS')
configure.in:140: warning: 'INCLUDES' is the old name for 'AM_CPPFLAGS' (or '*_CPPFLAGS')
lib/cqdb/Makefile.am:9: warning: source file 'src/lookup3.c' is in a subdirectory,
lib/cqdb/Makefile.am:9: but option 'subdir-objects' is disabled
automake: warning: possible forward-incompatibility.
automake: At least one source file is in a subdirectory, but the 'subdir-objects'
automake: automak

In [10]:
# Check the system and generate the Makefile (on Apple Silicon this needs --disable-sse2, see later cell)
!./configure #configure the package for the system. This will check for required libraries and create a Makefile.

checking build system type... aarch64-apple-darwin25.5.0
checking host system type... aarch64-apple-darwin25.5.0
checking for gcc... gcc
checking whether the C compiler works... yes
checking for C compiler default output file name... a.out
checking for suffix of executables... 
checking whether we are cross compiling... no
checking for suffix of object files... o
checking whether the compiler supports GNU C... yes
checking whether gcc accepts -g... yes
checking for gcc option to enable C23 features... -std=gnu23
checking whether gcc -std=gnu23 understands -c and -o together... yes
checking for stdio.h... yes
checking for stdlib.h... yes
checking for string.h... yes
checking for inttypes.h... yes
checking for stdint.h... yes
checking for strings.h... yes
checking for sys/stat.h... yes
checking for sys/types.h... yes
checking for unistd.h... yes
checking for wchar.h... yes
checking for minix/config.h... no
checking whether it is safe to define __EXTENSIONS__... yes
checking whether _XOPE

In [11]:
# Compile crfsuite (fails here on Apple Silicon with an SSE2 error until reconfigured below)
!make

/Library/Developer/CommandLineTools/usr/bin/make  all-recursive
Making all in include
make[2]: Nothing to be done for `all'.
Making all in lib/cqdb
make[2]: Nothing to be done for `all'.
Making all in lib/crf
/bin/sh ../../libtool  --tag=CC   --mode=compile gcc -std=gnu23 -DHAVE_CONFIG_H -I. -I../.. -I../.. -I../../include -I. -I../.. -I../../include -I.  -I../../lib/cqdb/include -mfpmath=sse -msse2 -DUSE_SSE -O3 -fomit-frame-pointer -ffast-math -Winline -std=c99  -MT libcrfsuite_la-dataset.lo -MD -MP -MF .deps/libcrfsuite_la-dataset.Tpo -c -o libcrfsuite_la-dataset.lo `test -f 'src/dataset.c' || echo './'`src/dataset.c
libtool: compile:  gcc -std=gnu23 -DHAVE_CONFIG_H -I. -I../.. -I../.. -I../../include -I. -I../.. -I../../include -I. -I../../lib/cqdb/include -mfpmath=sse -msse2 -DUSE_SSE -O3 -fomit-frame-pointer -ffast-math -Winline -std=c99 -MT libcrfsuite_la-dataset.lo -MD -MP -MF .deps/libcrfsuite_la-dataset.Tpo -c src/dataset.c  -fno-common -DPIC -o .libs/libcrfsuite_la-dataset.o

## liblbfgs Library Installation

In [12]:
# Move to /tmp to build the liblbfgs dependency separately
%cd /tmp

/private/tmp


In [13]:
# Clone liblbfgs, the L-BFGS optimization library crfsuite trains with
!git clone https://github.com/chokkan/liblbfgs

fatal: destination path 'liblbfgs' already exists and is not an empty directory.


In [14]:
# Move into the cloned liblbfgs folder
%cd liblbfgs/

/private/tmp/liblbfgs


In [15]:
# Generate liblbfgs's ./configure script
!./autogen.sh

configure.ac:102: warning: 'INCLUDES' is the old name for 'AM_CPPFLAGS' (or '*_CPPFLAGS')
lib/Makefile.am:24: warning: 'INCLUDES' is the old name for 'AM_CPPFLAGS' (or '*_CPPFLAGS')
sample/Makefile.am:15: warning: 'INCLUDES' is the old name for 'AM_CPPFLAGS' (or '*_CPPFLAGS')
configure.ac:48: warning: The macro 'AC_HEADER_STDC' is obsolete.
configure.ac:48: You should run autoupdate.
./lib/autoconf/headers.m4:664: AC_HEADER_STDC is expanded from...
configure.ac:48: the top level


In [16]:
# Check the system and generate liblbfgs's Makefile
!./configure

checking for a BSD-compatible install... /usr/bin/install -c
checking whether sleep supports fractional seconds... yes
checking filesystem timestamp resolution... 2
checking whether build environment is sane... yes
checking for a race-free mkdir -p... mkdir -p
checking for gawk... no
checking for mawk... no
checking for nawk... no
checking for awk... awk
checking whether make sets $(MAKE)... yes
checking whether make supports nested variables... yes
checking xargs -n works... yes
checking whether UID '501' is supported by ustar format... yes
checking whether GID '20' is supported by ustar format... yes
checking how to create a ustar tar archive... gnutar
checking whether to enable maintainer-specific portions of Makefiles... no
checking build system type... aarch64-apple-darwin25.5.0
checking host system type... aarch64-apple-darwin25.5.0
checking how to print strings... printf
checking whether make supports the include directive... yes (GNU style)
checking for gcc... gcc
checking whet

In [17]:
# Compile liblbfgs (run 'sudo make install' afterward in a terminal, not here, since sudo needs a password prompt)
!make

/Library/Developer/CommandLineTools/usr/bin/make  all-recursive
Making all in lib
/bin/sh ../libtool  --tag=CC   --mode=link gcc -std=gnu23 -O3 -ffast-math  -Wall -O3 -ffast-math  -Wall -no-undefined -release 1.10  -o liblbfgs.la -rpath /usr/local/lib lbfgs.lo  -lm 
libtool: link: gcc -std=gnu23 -dynamiclib  -o .libs/liblbfgs-1.10.dylib  .libs/lbfgs.o   -lm  -O3 -Wall -O3 -Wall   -install_name  /usr/local/lib/liblbfgs-1.10.dylib  
libtool: link: (cd ".libs" && rm -f "liblbfgs.dylib" && ln -s "liblbfgs-1.10.dylib" "liblbfgs.dylib")
libtool: link: ar cr .libs/liblbfgs.a lbfgs.o
libtool: link: ranlib .libs/liblbfgs.a
libtool: link: ( cd ".libs" && rm -f "liblbfgs.la" && ln -s "../liblbfgs.la" "liblbfgs.la" )
Making all in sample
gcc -std=gnu23 -DHAVE_CONFIG_H -I. -I.. -I.. -I../include   -O3 -ffast-math  -Wall -O3 -ffast-math  -Wall -MT sample.o -MD -MP -MF .deps/sample.Tpo -c -o sample.o sample.c
mv -f .deps/sample.Tpo .deps/sample.Po
/bin/sh ../libtool  --tag=CC   --mode=link gcc -std=g

```
ye@lst-hpc3090:/tmp/liblbfgs$ sudo make install
[sudo] password for ye:
Making install in lib
make[1]: Entering directory '/tmp/liblbfgs/lib'
make[2]: Entering directory '/tmp/liblbfgs/lib'
 /usr/bin/mkdir -p '/usr/local/lib'
 /bin/bash ../libtool   --mode=install /usr/bin/install -c   liblbfgs.la '/usr/local/lib'
libtool: install: /usr/bin/install -c .libs/liblbfgs-1.10.so /usr/local/lib/liblbfgs-1.10.so
libtool: install: (cd /usr/local/lib && { ln -s -f liblbfgs-1.10.so liblbfgs.so || { rm -f liblbfgs.so && ln -s liblbfgs-1.10.so liblbfgs.so; }; })
libtool: install: /usr/bin/install -c .libs/liblbfgs.lai /usr/local/lib/liblbfgs.la
libtool: install: /usr/bin/install -c .libs/liblbfgs.a /usr/local/lib/liblbfgs.a
libtool: install: chmod 644 /usr/local/lib/liblbfgs.a
libtool: install: ranlib /usr/local/lib/liblbfgs.a
libtool: finish: PATH="/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/snap/bin:/sbin" ldconfig -n /usr/local/lib
----------------------------------------------------------------------
Libraries have been installed in:
   /usr/local/lib

If you ever happen to want to link against installed libraries
in a given directory, LIBDIR, you must either use libtool, and
specify the full pathname of the library, or use the '-LLIBDIR'
flag during linking and do at least one of the following:
   - add LIBDIR to the 'LD_LIBRARY_PATH' environment variable
     during execution
   - add LIBDIR to the 'LD_RUN_PATH' environment variable
     during linking
   - use the '-Wl,-rpath -Wl,LIBDIR' linker flag
   - have your system administrator add LIBDIR to '/etc/ld.so.conf'

See any operating system documentation about shared libraries for
more information, such as the ld(1) and ld.so(8) manual pages.
----------------------------------------------------------------------
 /usr/bin/mkdir -p '/usr/local/include'
 /usr/bin/install -c -m 644 ../include/lbfgs.h '/usr/local/include'
make[2]: Leaving directory '/tmp/liblbfgs/lib'
make[1]: Leaving directory '/tmp/liblbfgs/lib'
Making install in sample
make[1]: Entering directory '/tmp/liblbfgs/sample'
make[2]: Entering directory '/tmp/liblbfgs/sample'
make[2]: Nothing to be done for 'install-exec-am'.
make[2]: Nothing to be done for 'install-data-am'.
make[2]: Leaving directory '/tmp/liblbfgs/sample'
make[1]: Leaving directory '/tmp/liblbfgs/sample'
make[1]: Entering directory '/tmp/liblbfgs'
make[2]: Entering directory '/tmp/liblbfgs'
make[2]: Nothing to be done for 'install-exec-am'.
 /usr/bin/mkdir -p '/usr/local/share/doc/liblbfgs'
 /usr/bin/install -c -m 644 README INSTALL COPYING AUTHORS ChangeLog NEWS '/usr/local/share/doc/liblbfgs'
make[2]: Leaving directory '/tmp/liblbfgs'
make[1]: Leaving directory '/tmp/liblbfgs'
ye@lst-hpc3090:/tmp/liblbfgs$
```

## Let's Do CRFSuite Installation Again

In [18]:
# Confirm current directory before going back to crfsuite
%pwd

'/private/tmp/liblbfgs'

In [19]:
# Move back into the crfsuite folder to rebuild it against liblbfgs
%cd crfsuite/

[Errno 2] No such file or directory: 'crfsuite/'
/private/tmp/liblbfgs


In [20]:
# Remove previous build artifacts so crfsuite rebuilds cleanly
!make clean

Making clean in lib
rm -f  liblbfgs.la
rm -f ./so_locations
rm -rf .libs _libs
rm -f *.o
rm -f *.lo
Making clean in sample
rm -rf .libs _libs
rm -f  sample
test -z "" || rm -f  sample
rm -f *.o
rm -f *.lo
rm -rf .libs _libs
rm -f *.lo


In [21]:
# Reconfigure crfsuite to link against the liblbfgs just installed to /usr/local
# --disable-sse2: crfsuite's configure.in unconditionally adds -msse2 (x86-only), which fails on Apple Silicon
!./configure --disable-sse2 CFLAGS="-I/usr/local/include -O3" LDFLAGS="-L/usr/local/lib" LIBS="-llbfgs"

checking for a BSD-compatible install... /usr/bin/install -c
checking whether sleep supports fractional seconds... yes
checking filesystem timestamp resolution... 2
checking whether build environment is sane... yes
checking for a race-free mkdir -p... mkdir -p
checking for gawk... no
checking for mawk... no
checking for nawk... no
checking for awk... awk
checking whether make sets $(MAKE)... yes
checking whether make supports nested variables... yes
checking xargs -n works... yes
checking whether UID '501' is supported by ustar format... yes
checking whether GID '20' is supported by ustar format... yes
checking how to create a ustar tar archive... gnutar
checking whether to enable maintainer-specific portions of Makefiles... no
checking build system type... aarch64-apple-darwin25.5.0
checking host system type... aarch64-apple-darwin25.5.0
checking how to print strings... printf
checking whether make supports the include directive... yes (GNU style)
checking for gcc... gcc
checking whet

In [22]:
# Rebuild crfsuite with liblbfgs support
!make

/Library/Developer/CommandLineTools/usr/bin/make  all-recursive
Making all in lib
/bin/sh ../libtool  --tag=CC   --mode=compile gcc -std=gnu23 -DHAVE_CONFIG_H -I. -I.. -I.. -I../include   -O3 -ffast-math -I/usr/local/include -O3 -Wall -O3 -ffast-math -I/usr/local/include -O3 -Wall -MT lbfgs.lo -MD -MP -MF .deps/lbfgs.Tpo -c -o lbfgs.lo lbfgs.c
libtool: compile:  gcc -std=gnu23 -DHAVE_CONFIG_H -I. -I.. -I.. -I../include -O3 -ffast-math -I/usr/local/include -O3 -Wall -O3 -ffast-math -I/usr/local/include -O3 -Wall -MT lbfgs.lo -MD -MP -MF .deps/lbfgs.Tpo -c lbfgs.c  -fno-common -DPIC -o .libs/lbfgs.o
libtool: compile:  gcc -std=gnu23 -DHAVE_CONFIG_H -I. -I.. -I.. -I../include -O3 -ffast-math -I/usr/local/include -O3 -Wall -O3 -ffast-math -I/usr/local/include -O3 -Wall -MT lbfgs.lo -MD -MP -MF .deps/lbfgs.Tpo -c lbfgs.c -o lbfgs.o >/dev/null 2>&1
mv -f .deps/lbfgs.Tpo .deps/lbfgs.Plo
/bin/sh ../libtool  --tag=CC   --mode=link gcc -std=gnu23 -O3 -ffast-math -I/usr/local/include -O3 -Wall -O

**make install** run ဖို့က sudo right ရှိမှရမယ်။

```
ye@lst-hpc3090:~/aif2/word-seg/tool/crfsuite$ sudo make install
Making install in include
make[1]: Entering directory '/home/ye/aif2/word-seg/tool/crfsuite/include'
make[2]: Entering directory '/home/ye/aif2/word-seg/tool/crfsuite/include'
make[2]: Nothing to be done for 'install-exec-am'.
 /usr/bin/mkdir -p '/usr/local/include'
 /usr/bin/install -c -m 644 crfsuite.h crfsuite_api.hpp crfsuite.hpp '/usr/local/include'
make[2]: Leaving directory '/home/ye/aif2/word-seg/tool/crfsuite/include'
make[1]: Leaving directory '/home/ye/aif2/word-seg/tool/crfsuite/include'
Making install in lib/cqdb
make[1]: Entering directory '/home/ye/aif2/word-seg/tool/crfsuite/lib/cqdb'
make[2]: Entering directory '/home/ye/aif2/word-seg/tool/crfsuite/lib/cqdb'
 /usr/bin/mkdir -p '/usr/local/lib'
 /bin/bash ../../libtool   --mode=install /usr/bin/install -c   libcqdb.la '/usr/local/lib'
libtool: install: /usr/bin/install -c .libs/libcqdb-0.12.so /usr/local/lib/libcqdb-0.12.so
libtool: install: (cd /usr/local/lib && { ln -s -f libcqdb-0.12.so libcqdb.so || { rm -f libcqdb.so && ln -s libcqdb-0.12.so libcqdb.so; }; })
libtool: install: /usr/bin/install -c .libs/libcqdb.lai /usr/local/lib/libcqdb.la
libtool: install: /usr/bin/install -c .libs/libcqdb.a /usr/local/lib/libcqdb.a
libtool: install: chmod 644 /usr/local/lib/libcqdb.a
libtool: install: ranlib /usr/local/lib/libcqdb.a
libtool: finish: PATH="/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/snap/bin:/sbin" ldconfig -n /usr/local/lib
----------------------------------------------------------------------
Libraries have been installed in:
   /usr/local/lib

If you ever happen to want to link against installed libraries
in a given directory, LIBDIR, you must either use libtool, and
specify the full pathname of the library, or use the '-LLIBDIR'
flag during linking and do at least one of the following:
   - add LIBDIR to the 'LD_LIBRARY_PATH' environment variable
     during execution
   - add LIBDIR to the 'LD_RUN_PATH' environment variable
     during linking
   - use the '-Wl,-rpath -Wl,LIBDIR' linker flag
   - have your system administrator add LIBDIR to '/etc/ld.so.conf'

See any operating system documentation about shared libraries for
more information, such as the ld(1) and ld.so(8) manual pages.
----------------------------------------------------------------------
make[2]: Nothing to be done for 'install-data-am'.
make[2]: Leaving directory '/home/ye/aif2/word-seg/tool/crfsuite/lib/cqdb'
make[1]: Leaving directory '/home/ye/aif2/word-seg/tool/crfsuite/lib/cqdb'
Making install in lib/crf
make[1]: Entering directory '/home/ye/aif2/word-seg/tool/crfsuite/lib/crf'
make[2]: Entering directory '/home/ye/aif2/word-seg/tool/crfsuite/lib/crf'
 /usr/bin/mkdir -p '/usr/local/lib'
 /bin/bash ../../libtool   --mode=install /usr/bin/install -c   libcrfsuite.la '/usr/local/lib'
libtool: warning: relinking 'libcrfsuite.la'
libtool: install: (cd /home/ye/aif2/word-seg/tool/crfsuite/lib/crf; /bin/bash "/home/ye/aif2/word-seg/tool/crfsuite/libtool"  --tag CC --mode=relink gcc -I../../lib/cqdb/include -mfpmath=sse -msse2 -DUSE_SSE -O3 -fomit-frame-pointer -ffast-math -Winline -std=c99 -I/usr/local/include -O3 -no-undefined -release 0.12 -o libcrfsuite.la -rpath /usr/local/lib libcrfsuite_la-dictionary.lo libcrfsuite_la-logging.lo libcrfsuite_la-params.lo libcrfsuite_la-quark.lo libcrfsuite_la-rumavl.lo libcrfsuite_la-dataset.lo libcrfsuite_la-holdout.lo libcrfsuite_la-train_arow.lo libcrfsuite_la-train_averaged_perceptron.lo libcrfsuite_la-train_l2sgd.lo libcrfsuite_la-train_lbfgs.lo libcrfsuite_la-train_passive_aggressive.lo libcrfsuite_la-crf1d_context.lo libcrfsuite_la-crf1d_model.lo libcrfsuite_la-crf1d_feature.lo libcrfsuite_la-crf1d_encode.lo libcrfsuite_la-crf1d_tag.lo libcrfsuite_la-crfsuite_train.lo libcrfsuite_la-crfsuite.lo ../../lib/cqdb/libcqdb.la -llbfgs -lm -llbfgs )
libtool: relink: gcc -shared  -fPIC -DPIC  .libs/libcrfsuite_la-dictionary.o .libs/libcrfsuite_la-logging.o .libs/libcrfsuite_la-params.o .libs/libcrfsuite_la-quark.o .libs/libcrfsuite_la-rumavl.o .libs/libcrfsuite_la-dataset.o .libs/libcrfsuite_la-holdout.o .libs/libcrfsuite_la-train_arow.o .libs/libcrfsuite_la-train_averaged_perceptron.o .libs/libcrfsuite_la-train_l2sgd.o .libs/libcrfsuite_la-train_lbfgs.o .libs/libcrfsuite_la-train_passive_aggressive.o .libs/libcrfsuite_la-crf1d_context.o .libs/libcrfsuite_la-crf1d_model.o .libs/libcrfsuite_la-crf1d_feature.o .libs/libcrfsuite_la-crf1d_encode.o .libs/libcrfsuite_la-crf1d_tag.o .libs/libcrfsuite_la-crfsuite_train.o .libs/libcrfsuite_la-crfsuite.o   -L/usr/local/lib -lcqdb -lm -llbfgs  -mfpmath=sse -msse2 -O3 -O3   -Wl,-soname -Wl,libcrfsuite-0.12.so -o .libs/libcrfsuite-0.12.so
libtool: install: /usr/bin/install -c .libs/libcrfsuite-0.12.soT /usr/local/lib/libcrfsuite-0.12.so
libtool: install: (cd /usr/local/lib && { ln -s -f libcrfsuite-0.12.so libcrfsuite.so || { rm -f libcrfsuite.so && ln -s libcrfsuite-0.12.so libcrfsuite.so; }; })
libtool: install: /usr/bin/install -c .libs/libcrfsuite.lai /usr/local/lib/libcrfsuite.la
libtool: install: /usr/bin/install -c .libs/libcrfsuite.a /usr/local/lib/libcrfsuite.a
libtool: install: chmod 644 /usr/local/lib/libcrfsuite.a
libtool: install: ranlib /usr/local/lib/libcrfsuite.a
libtool: finish: PATH="/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin:/snap/bin:/sbin" ldconfig -n /usr/local/lib
----------------------------------------------------------------------
Libraries have been installed in:
   /usr/local/lib

If you ever happen to want to link against installed libraries
in a given directory, LIBDIR, you must either use libtool, and
specify the full pathname of the library, or use the '-LLIBDIR'
flag during linking and do at least one of the following:
   - add LIBDIR to the 'LD_LIBRARY_PATH' environment variable
     during execution
   - add LIBDIR to the 'LD_RUN_PATH' environment variable
     during linking
   - use the '-Wl,-rpath -Wl,LIBDIR' linker flag
   - have your system administrator add LIBDIR to '/etc/ld.so.conf'

See any operating system documentation about shared libraries for
more information, such as the ld(1) and ld.so(8) manual pages.
----------------------------------------------------------------------
make[2]: Nothing to be done for 'install-data-am'.
make[2]: Leaving directory '/home/ye/aif2/word-seg/tool/crfsuite/lib/crf'
make[1]: Leaving directory '/home/ye/aif2/word-seg/tool/crfsuite/lib/crf'
Making install in frontend
make[1]: Entering directory '/home/ye/aif2/word-seg/tool/crfsuite/frontend'
make[2]: Entering directory '/home/ye/aif2/word-seg/tool/crfsuite/frontend'
 /usr/bin/mkdir -p '/usr/local/bin'
  /bin/bash ../libtool   --mode=install /usr/bin/install -c crfsuite '/usr/local/bin'
libtool: install: /usr/bin/install -c .libs/crfsuite /usr/local/bin/crfsuite
make[2]: Nothing to be done for 'install-data-am'.
make[2]: Leaving directory '/home/ye/aif2/word-seg/tool/crfsuite/frontend'
make[1]: Leaving directory '/home/ye/aif2/word-seg/tool/crfsuite/frontend'
Making install in swig
make[1]: Entering directory '/home/ye/aif2/word-seg/tool/crfsuite/swig'
make[2]: Entering directory '/home/ye/aif2/word-seg/tool/crfsuite/swig'
make[2]: Nothing to be done for 'install-exec-am'.
make[2]: Nothing to be done for 'install-data-am'.
make[2]: Leaving directory '/home/ye/aif2/word-seg/tool/crfsuite/swig'
make[1]: Leaving directory '/home/ye/aif2/word-seg/tool/crfsuite/swig'
make[1]: Entering directory '/home/ye/aif2/word-seg/tool/crfsuite'
make[2]: Entering directory '/home/ye/aif2/word-seg/tool/crfsuite'
make[2]: Nothing to be done for 'install-exec-am'.
 /usr/bin/mkdir -p '/usr/local/share/doc/crfsuite'
 /usr/bin/install -c -m 644 README INSTALL COPYING AUTHORS ChangeLog '/usr/local/share/doc/crfsuite'
make[2]: Leaving directory '/home/ye/aif2/word-seg/tool/crfsuite'
make[1]: Leaving directory '/home/ye/aif2/word-seg/tool/crfsuite'
ye@lst-hpc3090:~/aif2/word-seg/tool/crfsuite$
```

In [23]:
# Confirm working directory
%pwd


'/private/tmp/liblbfgs'

In [24]:
# Show crfsuite's top-level CLI usage (needs 'sudo make install' in a terminal so it's on PATH)
!crfsuite --help

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

USAGE: crfsuite <COMMAND> [OPTIONS]
    COMMAND     Command name to specify the processing
    OPTIONS     Arguments for the command (optional; command-specific)

COMMAND:
    learn       Obtain a model from a training set of instances
    tag         Assign suitable labels to given instances by using a model
    dump        Output a model in a plain-text format

For the usage of each command, specify -h option in the command argument.


In [25]:
# Show options for crfsuite's 'learn' (training) subcommand
!crfsuite learn --help

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

USAGE: crfsuite learn [OPTIONS] [DATA1] [DATA2] ...
Trains a model using training data set(s).

  DATA    file(s) corresponding to data set(s) for training; if multiple N files
          are specified, this utility assigns a group number (1...N) to the
          instances in each file; if a file name is '-', the utility reads a
          data set from STDIN

OPTIONS:
  -t, --type=TYPE       specify a graphical model (DEFAULT='1d'):
                        (this option is reserved for the future use)
      1d                    1st-order Markov CRF with state and transition
                            features; transition features are not conditioned
                            on observations
  -a, --algorithm=NAME  specify a training algorithm (DEFAULT='lbfgs')
      lbfgs                 L-BFGS with L1/L2 regularization
      l2sgd                 SGD with L2-regularization
      ap                    Averaged Perceptron
      

In [26]:
# Show options for crfsuite's 'tag' (prediction) subcommand
!crfsuite tag --help

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

USAGE: crfsuite tag [OPTIONS] [DATA]
Assign suitable labels to the instances in the data set given by a file (DATA).
If the argument DATA is omitted or '-', this utility reads a data from STDIN.
Evaluate the performance of the model on labeled instances (with -t option).

OPTIONS:
    -m, --model=MODEL   Read a model from a file (MODEL)
    -t, --test          Report the performance of the model on the data
    -r, --reference     Output the reference labels in the input data
    -p, --probability   Output the probability of the label sequences
    -i, --marginal      Output the marginal probabilitiy of items for their predicted label
    -l, --marginal-all  Output the marginal probabilities of items for all labels
    -q, --quiet         Suppress tagging results (useful for test mode)
    -h, --help          Show the usage of this command and exit


## Data Preparation

Github repository တစ်ခုလုံးကို download မလုပ်ပဲ လိုချင်တဲ့ ဖိုင်တစ်ဖိုင်ထဲကိုပဲ ကိုယ့်စက်ထဲကို ကော်ပီကူးချင်ရင် raw format အဖြစ်ပြောင်းကြည့်ပြီး ပုံမှန်အတိုင်း copy/paste လုပ်ယူလည်း ရတယ်။ သို့မဟုတ် အဲဒီ raw link ကို ယူလိုက်ပြီး wget command နဲ့လည်း ကော်ပီကူးယူလို့ ရပါတယ်။

In [27]:
# Confirm current directory before downloading corpus data
%pwd

'/private/tmp/liblbfgs'

In [28]:
# Move into the myPOS data folder (must exist first — create with 'mkdir -p' if missing)
%cd /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS

/Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS


In [29]:
#%cd ../../data/myPOS/ need to change it into real directory path

In [30]:
# Download the untagged Myanmar POS training corpus directly (skip cloning the whole myPOS repo)
!wget https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/mypos-ver.3.0.shuf.notag.nopunc.txt

--2026-07-30 20:21:44--  https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/mypos-ver.3.0.shuf.notag.nopunc.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8003::154, 2606:50c0:8000::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8003::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 7303817 (7.0M) [application/octet-stream]
Saving to: ‘mypos-ver.3.0.shuf.notag.nopunc.txt.2’

mypos-ver.3.0.shuf. 100%[===================>]   6.96M  2.35MB/s    in 3.0s    

2026-07-30 20:21:48 (2.35 MB/s) - ‘mypos-ver.3.0.shuf.notag.nopunc.txt.2’ saved [7303817/7303817]



In [31]:
# Preview the first few lines of the training corpus
!head mypos-ver.3.0.shuf.notag.nopunc.txt

၁၉၆၂ ခုနှစ် ခန့်မှန်း သန်းခေါင်စာရင်း အရ လူဦးရေ ၁၁၅၉၃၁ ယောက် ရှိ သည်
လူ တိုင်း တွင် သင့်မြတ် လျော်ကန် စွာ ကန့်သတ် ထား သည့် အလုပ် လုပ် ချိန် အပြင် လစာ နှင့်တကွ အခါ ကာလ အားလျော်စွာ သတ်မှတ် ထား သည့် အလုပ် အားလပ်ရက် များ ပါဝင် သည့် အနားယူခွင့် နှင့် အားလပ်ခွင့် ခံစားပိုင်ခွင့် ရှိ သည်
ဤ နည်း ကို စစ်ယူ သော နည်း ဟု ခေါ် သည်
စာပြန်ပွဲ ဆို တာ က အာဂုံဆောင် အလွတ်ကျက် ထား တဲ့ ပိဋကတ်သုံးပုံ စာပေ တွေ ကို စာစစ် သံဃာတော်ကြီး တွေ ရဲ့ ရှေ့ မှာ အလွတ် ပြန် ပြီး ရွတ်ပြ ရ တာ ပေါ့
ဒီ မှာ ကျွန်တော့် သက်သေခံကတ် ပါ
၂ဝ ရာစု မြန်မာ့ သမိုင်း သန်းဝင်းလှိုင် ၂ဝဝ၉ ခု မေ လ ကံကော်ဝတ်ရည် စာပေ
ကျွန်တော် မျက်မှန် တစ် လက် လုပ် ချင် ပါ တယ်
ကျွန်တော် တို့ က ဒီ အမှု ရဲ့ ကြံရာပါ ကို ဖမ်းမိ ဖို့ ကြိုးစား ခဲ့ တယ်
ကလေး မီးဖွား ဖို့ ခန့်မှန်း ရက် က ဘယ်တော့ ပါ လဲ
အရိုးရှင်းဆုံး ကာဗိုဟိုက်ဒရိတ် မှာ ဂလူးကို့စ် ဂလက်တို့စ် ဖရပ်တို့စ် စသည့် မိုနိုဆက်ကရိုက် များ ဖြစ် သည်


In [32]:
# Count lines/words/characters in the training corpus
!wc mypos-ver.3.0.shuf.notag.nopunc.txt

   43196  510695 7303817 mypos-ver.3.0.shuf.notag.nopunc.txt


In [33]:
# Download the test corpus (1k sentences)
!wget https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.nopipe.txt

--2026-07-30 20:21:49--  https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/refs/heads/master/corpus-ver-3.0/corpus/otest.1k.nopipe.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8003::154, 2606:50c0:8000::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8003::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 229758 (224K) [text/plain]
Saving to: ‘otest.1k.nopipe.txt.2’

otest.1k.nopipe.txt 100%[===================>] 224.37K  --.-KB/s    in 0.1s    

2026-07-30 20:21:49 (2.26 MB/s) - ‘otest.1k.nopipe.txt.2’ saved [229758/229758]



In [34]:
# Preview the test corpus
!head otest.1k.nopipe.txt

တစ်/tn ကိုက်/n ကို/ppm ဝမ်/n ခုနှစ်ထောင်/tn ပါ/part ။/punc
မနှစ်/n က/ppm သူ/pron ကျွန်မ/pron ကို/ppm သင်/v ပေး/part တယ်/ppm ။/punc
ကျွန်တော့်/pron ခုံ/n သွား/v ရှာ/v မလို့/part ။/punc
အတန်း/n စ/v တာ/part ကြာ/v ပြီ/ppm လား/part ။/punc
ဆေး/n နည်းနည်း/adv စား/v လိုက်/part ၊/punc သုံး/tn လေး/tn ရက်/n လောက်/part အနားယူ/v လိုက်/part ရင်/conj ပျောက်/v သွား/part မှာ/ppm ပါ/part ။/punc
အေးချမ်း/v မှု/part နဲ့/conj စည်းကမ်း/n ကို/ppm တည်မြဲ/v အောင်/part ထိန်းသိမ်း/v သည်/ppm ။/punc
ဇွန်း/n ကို/ppm လိုအပ်/v တယ်/ppm ။/punc
ဘွဲ့/n ရ/v ရင်/conj ဘာ/n လုပ်/v မ/part လို့/part လဲ/part ။/punc
ကျွန်တော်/pron ချောင်းဆိုး/v ခြင်း/part အတွက်/ppm တစ်/tn ခု/part ခု/part လို/v ချင်/part တယ်/ppm ။/punc
အသီးအနှံ/n တို့/part မှ/ppm လွဲ/v လျှင်/conj လူ/n တို့/part ၏/ppm အဓိက/n အစားအစာ/n မှာ/ppm ငါး/n ဖြစ်/v သည်/ppm ။/punc


In [35]:
# Count lines/words/characters in the test corpus
!wc otest.1k.nopipe.txt

    1000   13469  229758 otest.1k.nopipe.txt


In [36]:
%pwd

'/Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS'

In [37]:
!wget --help


GNU Wget 1.25.0, a non-interactive network retriever.
Usage: wget [OPTION]... [URL]...

Mandatory arguments to long options are mandatory for short options too.

Startup:
  -V,  --version                   display the version of Wget and exit
  -h,  --help                      print this help
  -b,  --background                go to background after startup
  -e,  --execute=COMMAND           execute a `.wgetrc'-style command

Logging and input file:
  -o,  --output-file=FILE          log messages to FILE
  -a,  --append-output=FILE        append messages to FILE
  -d,  --debug                     print lots of debugging information
  -q,  --quiet                     quiet (no output)
  -v,  --verbose                   be verbose (this is the default)
  -nv, --no-verbose                turn off verboseness, without being quiet
       --report-speed=TYPE         output bandwidth as TYPE.  TYPE can be bits
  -i,  --input-file=FILE           download URLs found in local or external FILE
  

In [38]:
!wget https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/f1a620ba271cae3d8658441eb626450fca09cdfa/corpus-draft-ver-1.0/mk-wordtag.pl -o mk-wordtag.pl

In [39]:
!ls -la mk-wordtag.pl
!head mk-wordtag.pl


-rw-r--r--@ 1 yadanar  staff  660 Jul 30 20:21 mk-wordtag.pl
--2026-07-30 20:21:50--  https://raw.githubusercontent.com/ye-kyaw-thu/myPOS/f1a620ba271cae3d8658441eb626450fca09cdfa/corpus-draft-ver-1.0/mk-wordtag.pl
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 2606:50c0:8003::154, 2606:50c0:8000::154, 2606:50c0:8002::154, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|2606:50c0:8003::154|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3967 (3.9K) [text/plain]
Saving to: ‘mk-wordtag.pl.1’

     0K ...                                                   100% 21.7M=0s

2026-07-30 20:21:50 (21.7 MB/s) - ‘mk-wordtag.pl.1’ saved [3967/3967]


In [40]:
!rm mk-wordtag.pl
!mv mk-wordtag.pl.1 mk-wordtag.pl

In [41]:
# Convert raw test text into word/tag pairs using mk-wordtag.pl (script must exist in the cwd — not included in this repo)
!perl ./mk-wordtag.pl ./otest.1k.nopipe.txt "\/" w > ./otest.txt 

In [42]:
# Preview the converted word/tag test file
!head ./otest.txt

တစ် ကိုက် ကို ဝမ် ခုနှစ်ထောင် ပါ ။
မနှစ် က သူ ကျွန်မ ကို သင် ပေး တယ် ။
ကျွန်တော့် ခုံ သွား ရှာ မလို့ ။
အတန်း စ တာ ကြာ ပြီ လား ။
ဆေး နည်းနည်း စား လိုက် ၊ သုံး လေး ရက် လောက် အနားယူ လိုက် ရင် ပျောက် သွား မှာ ပါ ။
အေးချမ်း မှု နဲ့ စည်းကမ်း ကို တည်မြဲ အောင် ထိန်းသိမ်း သည် ။
ဇွန်း ကို လိုအပ် တယ် ။
ဘွဲ့ ရ ရင် ဘာ လုပ် မ လို့ လဲ ။
ကျွန်တော် ချောင်းဆိုး ခြင်း အတွက် တစ် ခု ခု လို ချင် တယ် ။
အသီးအနှံ တို့ မှ လွဲ လျှင် လူ တို့ ၏ အဓိက အစားအစာ မှာ ငါး ဖြစ် သည် ။


## Tag Preparation for Word Segmentation

Based on paper: Win Pa Pa, Ye Kyaw Thu, Andrew Finch, Eiichiro Sumita, "Word Boundary Identification for Myanmar Text Using Conditional Random Fields", In Proceedings of the Ninth International Conference on Genetic and Evolutionary Computing (ICGEC 2015), August 26-28, 2015, Yangon, Myanmar, pp. 447-456.



In [43]:
# Show myWord_tagger.py's source (path is the instructor's machine; local copy is at codes/class-5/myWord_tagger.py)
!cat /Users/yadanar/Desktop/AIE-F-B2/codes/class-5/myWord_tagger.py

# myWordTagger.py
# Author: Thura Aung
# 9th December, 2023
# Data preparation for word segmentation CRF model training
# Ref: Win Pa Pa, Ye Kyaw Thu, Andrew Finch, Eiichiro Sumita, "Word Boundary Identification for Myanmar Text Using Conditional Random Fields", In Proceedings of the Ninth International Conference on Genetic and Evolutionary Computing (ICGEC 2015), August 26-28, 2015, Yangon, Myanmar, pp. 447-456.

import re
import argparse

parser = argparse.ArgumentParser(description='Data preparation for word segmentation')
parser.add_argument('-i', '--input', type=str, help='input file', required=True)
parser.add_argument('-m', '--mode', type=str, default=r's', help='s for syllable and c for character tagging', required=False)
parser.add_argument('-n', '--nTag', type=int, default=4, help='Number of Tags {2, 3, 4}', required=False)
args = parser.parse_args()

inputFile = getattr(args, 'input')
mode = getattr(args, 'mode')
n = getattr(args, 'nTag')

myConsonant = r"က-အ"
enChar = r"a-

## Bug found: myWord_tagger.py appends a junk line

`myWord_tagger.py` used three separate `if n == 2:` / `if n == 3:` / `if n == 4:` statements
(not `if/elif/elif`), with `else: print("Only 2 to 4 accepted.")` attached only to the last one.
So running with `-n 2` or `-n 3` correctly wrote the tagged data, but then the unrelated `else`
also fired (since `n != 4`), appending `"Only 2 to 4 accepted."` as one extra garbage line at the
end of the output file. That line has no tab/tag, which breaks `chunking.py` downstream.

**Fix applied** (in `codes/class-5/myWord_tagger.py`): changed the three `if`s into
`if / elif / elif / else`, so `else` only fires when `n` is genuinely not 2, 3, or 4.
If you already generated `train.tag2.txt`, `train.tag3.txt`, or `test.tag3.txt` before this fix,
re-run the tagging cells to regenerate them without the junk line.

## --help Calling

In [44]:
# Show myWord_tagger.py's command-line options
!python /Users/yadanar/Desktop/AIE-F-B2/codes/class-5/myWord_tagger.py --help

usage: myWord_tagger.py [-h] -i INPUT [-m MODE] [-n NTAG]

Data preparation for word segmentation

options:
  -h, --help            show this help message and exit
  -i INPUT, --input INPUT
                        input file
  -m MODE, --mode MODE  s for syllable and c for character tagging
  -n NTAG, --nTag NTAG  Number of Tags {2, 3, 4}


In [45]:
# Move into the myPOS data folder (instructor's path — remap to your local project path)
#%cd /home/ye/aif2/word-seg/data/myPOS
%cd /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS

/Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS


In [46]:
# Create a 'tag' folder to hold tagged output files
!mkdir tag

mkdir: tag: File exists


In [47]:
# Move into the 'tag' folder
%cd tag

/Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag


In [48]:
# Tag the training data in syllable mode with n=2 context
!python /Users/yadanar/Desktop/AIE-F-B2/codes/class-5/myWord_tagger.py --input /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/mypos-ver.3.0.shuf.notag.nopunc.txt --mode s -n 2 > /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/train.tag2.txt

In [49]:
# Preview the n=2 tagged training data
!head -n 30  /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/train.tag2.txt

၁	-
၉	-
၆	-
၂	|
ခု	-
နှစ်	|
ခန့်	-
မှန်း	|
သန်း	-
ခေါင်	-
စာ	-
ရင်း	|
အ	-
ရ	|
လူ	-
ဦး	-
ရေ	|
၁	-
၁	-
၅	-
၉	-
၃	-
၁	|
ယောက်	|
ရှိ	|
သည်	|

လူ	|
တိုင်း	|
တွင်	|


## Syllable, 3 tag Format

In [50]:
# Tag the training data again with n=3 context (3-tag format)
!python /Users/yadanar/Desktop/AIE-F-B2/codes/class-5/myWord_tagger.py --input /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/mypos-ver.3.0.shuf.notag.nopunc.txt --mode s -n 3 > /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/train.tag3.txt

In [51]:
# Preview the n=3 tagged training data
!head -n 50 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/train.tag3.txt

၁	<
၉	-
၆	-
၂	|
ခု	<
နှစ်	|
ခန့်	<
မှန်း	|
သန်း	<
ခေါင်	-
စာ	-
ရင်း	|
အ	<
ရ	|
လူ	<
ဦး	-
ရေ	|
၁	<
၁	-
၅	-
၉	-
၃	-
၁	|
ယောက်	|
ရှိ	|
သည်	|

လူ	|
တိုင်း	|
တွင်	|
သင့်	<
မြတ်	|
လျော်	<
ကန်	|
စွာ	|
ကန့်	<
သတ်	|
ထား	|
သည့်	|
အ	<
လုပ်	|
လုပ်	|
ချိန်	|
အ	<
ပြင်	|
လ	<
စာ	|
နှင့်	<
တ	-
ကွ	|


## Tagging for Test Data

In [52]:
# Tag the test data the same way (n=3) for consistency with training
!python /Users/yadanar/Desktop/AIE-F-B2/codes/class-5/myWord_tagger.py --input /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/otest.txt --mode s -n 3 > /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/test.tag3.txt

In [ ]:
# Preview the tagged test data
!head -n 100 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/test.tag3.txt

တစ်	|
ကိုက်	|
ကို	|
ဝမ်	|
ခု	<
နှစ်	-
ထောင်	|
ပါ	|
။	|

မ	<
နှစ်	|
က	|
သူ	|
ကျွန်	<
မ	|
ကို	|
သင်	|
ပေး	|
တယ်	|
။	|

ကျွန်	<
တော့်	|
ခုံ	|
သွား	|
ရှာ	|
မ	<
လို့	|
။	|

အ	<
တန်း	|
စ	|
တာ	|
ကြာ	|
ပြီ	|
လား	|
။	|

ဆေး	|
နည်း	<
နည်း	|
စား	|
လိုက်	|
၊	|
သုံး	|
လေး	|
ရက်	|
လောက်	|
အ	<
နား	-
ယူ	|
လိုက်	|
ရင်	|
ပျောက်	|
သွား	|
မှာ	|
ပါ	|
။	|

အေး	<
ချမ်း	|
မှု	|
နဲ့	|
စည်း	<
ကမ်း	|
ကို	|
တည်	<
မြဲ	|
အောင်	|
ထိန်း	<
သိမ်း	|
သည်	|
။	|

ဇွန်း	|
ကို	|
လို	<
အပ်	|
တယ်	|
။	|

ဘွဲ့	|
ရ	|
ရင်	|
ဘာ	|
လုပ်	|
မ	|
လို့	|
လဲ	|
။	|

ကျွန်	<
တော်	|
ချောင်း	<
ဆိုး	|
ခြင်း	|
အ	<
တွက်	|


## Change into CRFSuite Data Format

In [ ]:
# Move into crfsuite's example folder (has chunking.py, the CRFsuite feature-format template)
%cd /Users/yadanar/Desktop/AIE-F-B2/notebooks/tool/crfsuite/example

In [55]:
# List example folder contents to confirm chunking.py is there
!ls

__pycache__        crfutils.py        pos.py             train.crfsuite.txt
bk                 debug.crfsuite.txt template.py
chunking.py        ner.py             tool


for Word segmentation, we need to use chunking.py code. 

In [56]:
# Convert a small debug sample into CRFsuite's training format using chunking.py
!cat /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/debug.tag3.txt | python ./chunking.py > debug.crfsuite.txt

cat: /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/debug.tag3.txt: No such file or directory


In [57]:
# Preview the converted CRFsuite-format debug data
!cat debug.crfsuite.txt

## We Found Error! :)

```
ဘူ	<
တာ	-
ရုံ	|
က	|
အ	<
လွန်	-
တ	-
ရာ	|
ပြ	<
ည့်	-
ကျပ်	|
နေ	|
သည်	|
။	|

Only 2 to 4 accepted.
```

## Backup Original Code

In [58]:
# Create a backup folder before modifying chunking.py
!mkdir bk

mkdir: bk: File exists


In [59]:
# Back up the original chunking.py before editing it
!cp chunking.py ./bk/

In [60]:
# Confirm the backup was copied
!ls ./bk/

chunking.py


## Code Updating

In [61]:
# Confirm current directory before editing chunking.py
%pwd

'/Users/yadanar/Desktop/AIE-F-B2/notebooks/tool/crfsuite/example'

In [62]:
# View chunking.py's source (the feature-extraction template) before editing
!cat ./chunking.py

#!/usr/bin/env python

"""
A feature extractor for chunking.
Copyright 2010,2011 Naoaki Okazaki.
"""

# Separator of field values.
separator = ' '

# Field names of the input data.
fields = 'w pos y'

# Attribute templates.
templates = (
    (('w', -2), ),
    (('w', -1), ),
    (('w',  0), ),
    (('w',  1), ),
    (('w',  2), ),
    (('w', -1), ('w',  0)),
    (('w',  0), ('w',  1)),
    (('pos', -2), ),
    (('pos', -1), ),
    (('pos',  0), ),
    (('pos',  1), ),
    (('pos',  2), ),
    (('pos', -2), ('pos', -1)),
    (('pos', -1), ('pos',  0)),
    (('pos',  0), ('pos',  1)),
    (('pos',  1), ('pos',  2)),
    (('pos', -2), ('pos', -1), ('pos',  0)),
    (('pos', -1), ('pos',  0), ('pos',  1)),
    (('pos',  0), ('pos',  1), ('pos',  2)),
    )


import crfutils

def feature_extractor(X):
    # Apply attribute templates to obtain features (in fact, attributes)
    crfutils.apply_templates(X, templates)
    if X:
	# Append BOS and EOS features manually
        X[0]['F'].append('

## Bug found: chunking.py's separator and templates were never fully updated

`codes/class-5/notes.txt` documents 3 required edits to `chunking.py` for this 2-column
(syllable, tag) word-segmentation data, but only one had actually been applied:

1. `fields = 'w y'` — **done**
2. `separator = '\t'` — **was still `' '` (a space)**, causing every line to fail to split
   correctly (your tag files are tab-separated), e.g. `ValueError: Too few fields (1) for ['w', 'y']`
   on line 1.
3. `templates` — **still had the full CoNLL2000 template list**, including entries referencing a
   `'pos'` field that doesn't exist in this data (only `'w'` and `'y'` do). This would crash with
   a `KeyError` once the separator issue above was fixed.

**Fix applied**: `separator` changed to `'\t'`, and `templates` trimmed to just the 7 `'w'`-based
entries from `notes.txt`. Verified end-to-end: `chunking.py` now exits cleanly and produces valid
CRFsuite-format output from a freshly re-tagged file.

In [63]:
# Convert the training data into CRFsuite format using chunking.py
!cat /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/train.tag3.txt | python ./chunking.py > train.crfsuite.txt

Traceback (most recent call last):
  File "/Users/yadanar/Desktop/AIE-F-B2/notebooks/tool/crfsuite/example/./chunking.py", line 49, in <module>
    crfutils.main(feature_extractor, fields=fields, sep=separator)
  File "/Users/yadanar/Desktop/AIE-F-B2/notebooks/tool/crfsuite/example/crfutils.py", line 160, in main
    for X in readiter(fi, F, options.separator):
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/yadanar/Desktop/AIE-F-B2/notebooks/tool/crfsuite/example/crfutils.py", line 64, in readiter
    raise ValueError(
ValueError: Too few fields (1) for ['w', 'pos', 'y']
၁	<
cat: stdout: Broken pipe


## Change into CRFSuite Data Format Again

In [ ]:
# Make sure we're in crfsuite's example folder before calling chunking.py
%cd /Users/yadanar/Desktop/AIE-F-B2/notebooks/tool/crfsuite/example

In [106]:
# Regenerate train.tag2.txt, train.tag3.txt, test.tag3.txt with the fixed myWord_tagger.py
# (myWord_tagger.py's if/elif bug is fixed now, so these no longer get a junk trailing line)
!python /Users/yadanar/Desktop/AIE-F-B2/codes/class-5/myWord_tagger.py --input /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/mypos-ver.3.0.shuf.notag.nopunc.txt --mode s -n 2 > /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/train.tag2.txt
!python /Users/yadanar/Desktop/AIE-F-B2/codes/class-5/myWord_tagger.py --input /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/mypos-ver.3.0.shuf.notag.nopunc.txt --mode s -n 3 > /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/train.tag3.txt
!python /Users/yadanar/Desktop/AIE-F-B2/codes/class-5/myWord_tagger.py --input /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/otest.txt --mode s -n 3 > /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/test.tag3.txt

In [110]:
# Convert the test data into CRFsuite format
!cat /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/test.tag3.txt | python ./chunking.py > /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/test.crfsuite.txt

In [111]:
# Preview the converted test data
!head -n 30 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/test.crfsuite.txt

|	w[0]=တစ်	w[1]=ကိုက်	w[2]=ကို	w[0]|w[1]=တစ်|ကိုက်	__BOS__
|	w[-1]=တစ်	w[0]=ကိုက်	w[1]=ကို	w[2]=ဝမ်	w[-1]|w[0]=တစ်|ကိုက်	w[0]|w[1]=ကိုက်|ကို
|	w[-2]=တစ်	w[-1]=ကိုက်	w[0]=ကို	w[1]=ဝမ်	w[2]=ခု	w[-1]|w[0]=ကိုက်|ကို	w[0]|w[1]=ကို|ဝမ်
|	w[-2]=ကိုက်	w[-1]=ကို	w[0]=ဝမ်	w[1]=ခု	w[2]=နှစ်	w[-1]|w[0]=ကို|ဝမ်	w[0]|w[1]=ဝမ်|ခု
<	w[-2]=ကို	w[-1]=ဝမ်	w[0]=ခု	w[1]=နှစ်	w[2]=ထောင်	w[-1]|w[0]=ဝမ်|ခု	w[0]|w[1]=ခု|နှစ်
-	w[-2]=ဝမ်	w[-1]=ခု	w[0]=နှစ်	w[1]=ထောင်	w[2]=ပါ	w[-1]|w[0]=ခု|နှစ်	w[0]|w[1]=နှစ်|ထောင်
|	w[-2]=ခု	w[-1]=နှစ်	w[0]=ထောင်	w[1]=ပါ	w[2]=။	w[-1]|w[0]=နှစ်|ထောင်	w[0]|w[1]=ထောင်|ပါ
|	w[-2]=နှစ်	w[-1]=ထောင်	w[0]=ပါ	w[1]=။	w[-1]|w[0]=ထောင်|ပါ	w[0]|w[1]=ပါ|။
|	w[-2]=ထောင်	w[-1]=ပါ	w[0]=။	w[-1]|w[0]=ပါ|။	__EOS__

<	w[0]=မ	w[1]=နှစ်	w[2]=က	w[0]|w[1]=မ|နှစ်	__BOS__
|	w[-1]=မ	w[0]=နှစ်	w[1]=က	w[2]=သူ	w[-1]|w[0]=မ|နှစ်	w[0]|w[1]=နှစ်|က
|	w[-2]=မ	w[-1]=နှစ်	w[0]=က	w[1]=သူ	w[2]=ကျွန်	w[-1]|w[0]=နှစ်|က	w[0]|w[1]=က|သူ
|	w[-2]=နှစ်	w[-1]=က	w[0]=သူ	w[1]=ကျွန်	w[2]=မ	w[-1]|w[0]=က|သူ	w[0]|w[1]=သူ|ကျွန်
<	w[

Format changing for training data: 

In [112]:
# Regenerate the CRFsuite-format training data (writing into the tag/data folder)
!cat /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/train.tag3.txt | python ./chunking.py > /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/train.crfsuite.txt

In [113]:
# Preview the CRFsuite-format training data
!head -n 30 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/train.crfsuite.txt

<	w[0]=၁	w[1]=၉	w[2]=၆	w[0]|w[1]=၁|၉	__BOS__
-	w[-1]=၁	w[0]=၉	w[1]=၆	w[2]=၂	w[-1]|w[0]=၁|၉	w[0]|w[1]=၉|၆
-	w[-2]=၁	w[-1]=၉	w[0]=၆	w[1]=၂	w[2]=ခု	w[-1]|w[0]=၉|၆	w[0]|w[1]=၆|၂
|	w[-2]=၉	w[-1]=၆	w[0]=၂	w[1]=ခု	w[2]=နှစ်	w[-1]|w[0]=၆|၂	w[0]|w[1]=၂|ခု
<	w[-2]=၆	w[-1]=၂	w[0]=ခု	w[1]=နှစ်	w[2]=ခန့်	w[-1]|w[0]=၂|ခု	w[0]|w[1]=ခု|နှစ်
|	w[-2]=၂	w[-1]=ခု	w[0]=နှစ်	w[1]=ခန့်	w[2]=မှန်း	w[-1]|w[0]=ခု|နှစ်	w[0]|w[1]=နှစ်|ခန့်
<	w[-2]=ခု	w[-1]=နှစ်	w[0]=ခန့်	w[1]=မှန်း	w[2]=သန်း	w[-1]|w[0]=နှစ်|ခန့်	w[0]|w[1]=ခန့်|မှန်း
|	w[-2]=နှစ်	w[-1]=ခန့်	w[0]=မှန်း	w[1]=သန်း	w[2]=ခေါင်	w[-1]|w[0]=ခန့်|မှန်း	w[0]|w[1]=မှန်း|သန်း
<	w[-2]=ခန့်	w[-1]=မှန်း	w[0]=သန်း	w[1]=ခေါင်	w[2]=စာ	w[-1]|w[0]=မှန်း|သန်း	w[0]|w[1]=သန်း|ခေါင်
-	w[-2]=မှန်း	w[-1]=သန်း	w[0]=ခေါင်	w[1]=စာ	w[2]=ရင်း	w[-1]|w[0]=သန်း|ခေါင်	w[0]|w[1]=ခေါင်|စာ
-	w[-2]=သန်း	w[-1]=ခေါင်	w[0]=စာ	w[1]=ရင်း	w[2]=အ	w[-1]|w[0]=ခေါင်|စာ	w[0]|w[1]=စာ|ရင်း
|	w[-2]=ခေါင်	w[-1]=စာ	w[0]=ရင်း	w[1]=အ	w[2]=ရ	w[-1]|w[0]=စာ|ရင်း	w[0]|w[1]=ရင်း|အ
<	w[-2]=စာ	w[-1]=ရင်း	w[0]=

## Training CRF Model for Word Segmentation

In [114]:
# Train a CRF model on the word-segmentation training data, saved as ws_tag3.model
!time crfsuite learn -m ws_tag3.model /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/train.crfsuite.txt

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

Start time of the training: 2026-07-30T14:38:30Z

Reading the data set(s)
[1] /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/train.crfsuite.txt
0....1....2....3....4....5....6....7....8....9....10
Number of instances: 43197
Seconds required: 2.430

Statistics the data set(s)
Number of data sets (groups): 1
Number of instances: 43196
Number of items: 776188
Number of attributes: 229827
Number of labels: 3

Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 0
0....1....2....3....4....5....6....7....8....9....10
Number of features: 272897
Seconds required: 0.661

L-BFGS optimization
c1: 0.000000
c2: 1.000000
num_memories: 6
max_iterations: 2147483647
epsilon: 0.000010
stop: 10
delta: 0.000010
linesearch: MoreThuente
linesearch.max_iterations: 20

***** Iteration #1 *****
Loss: 671164.745922
Feature norm: 1.000000
Error norm: 98468.199914
Active features: 272897

In [115]:
# Train again with -e2: evaluate against the held-out test set every 2 iterations
!time crfsuite learn -e2 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/train.crfsuite.txt /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/test.crfsuite.txt

CRFSuite 0.12.2  Copyright (c) 2007-2013 Naoaki Okazaki

Start time of the training: 2026-07-30T14:40:28Z

Reading the data set(s)
[1] /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/train.crfsuite.txt
0....1....2....3....4....5....6....7....8....9....10
Number of instances: 43197
Seconds required: 2.432
[2] /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/test.crfsuite.txt
0....1....2....3....4....5....6....7....8....9....10
Number of instances: 1001
Seconds required: 0.062

Statistics the data set(s)
Number of data sets (groups): 2
Number of instances: 44196
Number of items: 796190
Number of attributes: 230564
Number of labels: 3

Holdout group: 2

Feature generation
type: CRF1d
feature.minfreq: 0.000000
feature.possible_states: 0
feature.possible_transitions: 0
0....1....2....3....4....5....6....7....8....9....10
Number of features: 272897
Seconds required: 0.698

L-BFGS optimization
c1: 0.000000
c2: 1.000000
num_memories: 6
max_iterations: 2147483647
epsilon: 0.000

## Tagging with Trained Model

In [116]:
# Apply the trained model to tag the test data
!time crfsuite tag -m ws_tag3.model /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/test.crfsuite.txt > /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/test.tag.out

crfsuite tag -m ws_tag3.model  >   0.03s user 0.00s system 92% cpu 0.037 total


In [117]:
# Preview the model's tagging output
!head -n 50 /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/test.tag.out

|
|
|
|
<
|
|
|
|

<
|
|
|
<
|
|
|
|
|
|

<
|
|
|
|
<
|
|

<
|
|
|
|
|
|
|

|
<
|
|
|
|
|
|
|
|


## Tagging and Compare with Reference Tags

Left side is reference, right side is hypothesis. 

In [118]:
# Tag test data with -r to print reference and hypothesis tags side by side
!time crfsuite tag -r -m ws_tag3.model /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/test.crfsuite.txt > ref_hyp.out.txt

crfsuite tag -r -m ws_tag3.model  > ref_hyp.out.txt  0.03s user 0.00s system 88% cpu 0.040 total


In [119]:
# Preview the reference-vs-hypothesis comparison
!head -n 50 ./ref_hyp.out.txt

|	|
|	|
|	|
|	|
<	<
-	|
|	|
|	|
|	|

<	<
|	|
|	|
|	|
<	<
|	|
|	|
|	|
|	|
|	|
|	|

<	<
|	|
|	|
|	|
|	|
<	<
|	|
|	|

<	<
|	|
|	|
|	|
|	|
|	|
|	|
|	|

|	|
<	<
|	|
|	|
|	|
|	|
|	|
|	|
|	|
|	|


## Check Tagwise Performance

In [120]:
# Tag test data in quiet mode (-qt) to print only the accuracy summary
!time crfsuite tag -qt -m ./ws_tag3.model /Users/yadanar/Desktop/AIE-F-B2/notebooks/data/myPOS/tag/test.crfsuite.txt 

Performance by label (#match, #model, #ref) (precision, recall, F1):
    <: (4214, 4453, 4364) (0.9463, 0.9656, 0.9559)
    -: (2004, 2148, 2170) (0.9330, 0.9235, 0.9282)
    |: (13187, 13401, 13468) (0.9840, 0.9791, 0.9816)
Macro-average precision, recall, F1: (0.954440, 0.956089, 0.955222)
Item accuracy: 19405 / 20002 (0.9702)
Instance accuracy: 700 / 1000 (0.7000)
Elapsed time: 0.029409 [sec] (34003.2 [instance/sec])
crfsuite tag -qt -m ./ws_tag3.model   0.03s user 0.00s system 96% cpu 0.035 total


## Dump the Trained Model

In [121]:
# Confirm current directory
%pwd

'/Users/yadanar/Desktop/AIE-F-B2/notebooks/tool/crfsuite/example'

In [122]:
# Check the trained model file's size
!ls ws_tag3.model -hl

ls: -hl: No such file or directory
ws_tag3.model


In [123]:
# Dump the model's learned feature weights (first 200 lines) to inspect them
!crfsuite dump ./ws_tag3.model | head -n 200

FILEHEADER = {
  magic: lCRF
  size: 22010544
  type: FOMC
  version: 100
  num_features: 0
  num_labels: 3
  num_attrs: 229827
  off_features: 0x30
  off_labels: 0x534850
  off_attrs: 0x5350C2
  off_labelrefs: 0x123245C
  off_attrrefs: 0x12324A0
}

LABELS = {
      0: <
      1: -
      2: |
}

ATTRIBUTES = {
      0: w[0]=၁
      1: w[1]=၉
      2: w[2]=၆
      3: w[0]|w[1]=၁|၉
      4: __BOS__
      5: w[-1]=၁
      6: w[0]=၉
      7: w[1]=၆
      8: w[2]=၂
      9: w[-1]|w[0]=၁|၉
     10: w[0]|w[1]=၉|၆
     11: w[-2]=၁
     12: w[-1]=၉
     13: w[0]=၆
     14: w[1]=၂
     15: w[2]=ခု
     16: w[-1]|w[0]=၉|၆
     17: w[0]|w[1]=၆|၂
     18: w[-2]=၉
     19: w[-1]=၆
     20: w[0]=၂
     21: w[1]=ခု
     22: w[2]=နှစ်
     23: w[-1]|w[0]=၆|၂
     24: w[0]|w[1]=၂|ခု
     25: w[-2]=၆
     26: w[-1]=၂
     27: w[0]=ခု
     28: w[1]=နှစ်
     29: w[2]=ခန့်
     30: w[-1]|w[0]=၂|ခု
     31: w[0]|w[1]=ခု|နှစ်
     32: w[-2]=၂
     33: w[-1]=ခု
     34: w[0]=နှစ်
     35: w[1]=ခန့်
     36: w

In [124]:
# Dump the full trained model to a text file
!crfsuite dump ./ws_tag3.model > ./ws_tag3.model.txt

In [125]:
# Preview the end of the dumped model (state-transition weights)
!tail -n 200 ./ws_tag3.model.txt

  (0) w[0]|w[1]=လျှင်|ရှေ့ --> |: 0.001953
  (0) w[-1]|w[0]=လျှင်|ရှေ့ --> |: 0.345880
  (0) w[0]|w[1]=ရှေ့|ခြေ --> |: 0.345880
  (0) w[-1]|w[0]=ရှေ့|ခြေ --> |: 0.012375
  (0) w[0]|w[1]=ရွက်|လာ --> |: 0.005399
  (0) w[-1]|w[0]=ရွက်|လာ --> |: 0.000305
  (0) w[0]|w[1]=သောက်|ချောင်း --> |: 0.020729
  (0) w[-1]|w[0]=သောက်|ချောင်း --> <: 0.098925
  (0) w[0]|w[1]=ကဲ|ဦး --> |: 0.057755
  (0) w[-1]|w[0]=ကဲ|ဦး --> |: 0.025783
  (0) w[0]|w[1]=ကျော်|တ --> |: 0.229723
  (0) w[-1]|w[0]=ကျော်|တ --> <: 0.235816
  (0) w[0]|w[1]=ဘက်|ခြုံ --> |: 0.116774
  (0) w[-1]|w[0]=ဘက်|ခြုံ --> |: 0.037922
  (0) w[0]|w[1]=ထား|ဗျို့ --> |: 0.051340
  (0) w[-1]|w[0]=ထား|ဗျို့ --> |: 0.126342
  (0) w[0]|w[1]=ဗျို့|နား --> |: 0.126342
  (0) w[-1]|w[0]=ဗျို့|နား --> |: 0.047757
  (0) w[0]|w[1]=နား|နှစ် --> |: 0.047757
  (0) w[-1]|w[0]=နား|နှစ် --> |: 0.089507
  (0) w[0]|w[1]=အောင်|ခြုံ --> |: 0.213953
  (0) w[-1]|w[0]=အောင်|ခြုံ --> |: 0.099372
  (0) w[0]|w[1]=ခြုံ|နော် --> |: 0.099372
  (0) w[-1]|w[0]=ခြုံ|နော် --> 